# 03 · Experimentos de embeddings e recuperação

Compara, em um conjunto de perguntas, a recuperação **densa**, **BM25** e **híbrida (RRF)**,
e verifica se os prefixos `query:`/`passage:` do E5 estão sendo aplicados.

> Requer o índice: `python scripts/build_index.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 200)

In [2]:
from voc_rag.config import get_settings
from voc_rag.indexing import SentenceTransformerEmbedder, VectorStore
from voc_rag.retrieval import HybridRetriever, process_query

settings = get_settings()
store = VectorStore.load(settings.vectorstore_dir)
embedder = SentenceTransformerEmbedder(settings.embedding_model)
hybrid = HybridRetriever(store, embedder, use_bm25=True)
dense_only = HybridRetriever(store, embedder, use_bm25=False)
print(store.manifest)

{'embedding_model': 'intfloat/multilingual-e5-small', 'n_documents': 41951, 'dimension': 384, 'created_at': '2026-11-12T12:31:02+00:00'}


In [3]:
def show(question, retriever, k=8, auto_filters=False):
    pq = process_query(question, retriever.known_categories, auto_filters=auto_filters)
    cands, stats = retriever.retrieve(pq, k=40)
    rows = [{**store.documents.iloc[c.position][["text", "review_score"]].to_dict(),
             "dense": round(c.dense_score, 3), "bm25_rank": c.bm25_rank, "dense_rank": c.dense_rank}
            for c in cands[:k]]
    return pd.DataFrame(rows)

q = "O produto não foi entregue até hoje"
display(show(q, dense_only))
display(show(q, hybrid))

,text,review_score,dense,bm25_rank,dense_rank
0,Recebi hoje,5,0.926,None,1
1,ENTREGA ANTES DO PRAZO,5,0.918,None,2
2,Esperando a entrega,1,0.917,None,3
3,DEmora na entrega,2,0.917,None,4
4,Demora a entrega,1,0.915,None,5
5,Ainda agreo a entrega,3,0.915,None,6
6,ENTREGA REALIZADA ANTES DO PRAZO,5,0.913,None,7
7,entrega antes do previsto,5,0.913,None,8


,text,review_score,dense,bm25_rank,dense_rank
0,Recebi hoje,5,0.926,NaN,1.0
1,Só atrasou por causa dos correios,4,0.853,1.0,NaN
2,ENTREGA ANTES DO PRAZO,5,0.918,NaN,2.0
3,bom dia não sei se foi os correios que atrasou mas o produto demorou demais para chegar,1,0.853,2.0,NaN
4,Esperando a entrega,1,0.917,NaN,3.0
5,"A entrega atrasou, mas penso que foi por culpa dos Correios... Não sei...",3,0.839,3.0,NaN
6,DEmora na entrega,2,0.917,NaN,4.0
7,"Produto não foi entregue no prazo, mas isso aconteceu por causa do atraso do correios.",5,0.854,4.0,NaN


## Distribuição de similaridade: pergunta no escopo vs. fora do escopo

In [4]:
import numpy as np
questions = {
    "no escopo": ["Problemas com a entrega", "Produto veio com defeito", "Elogios à qualidade",
                  "Atendimento do vendedor", "Produto diferente do anunciado"],
    "fora do escopo": ["Qual é a capital da Austrália?", "Receita de bolo de chocolate",
                       "Cotação do bitcoin", "Escreva uma função em Python", "Quem ganhou a Copa de 2002?"],
}
rows = []
for group, qs in questions.items():
    for q in qs:
        sims = store.embeddings @ embedder.embed_query(q)
        top = np.sort(sims)[::-1]
        rows.append({"grupo": group, "pergunta": q, "top1": top[0], "top10": top[9], "top50": top[49]})
sim_df = pd.DataFrame(rows).round(3)
sim_df

,grupo,pergunta,top1,top10,top50
0,no escopo,Problemas com a entrega,0.947,0.935,0.917
1,no escopo,Produto veio com defeito,0.978,0.958,0.936
2,no escopo,Elogios à qualidade,0.930,0.924,0.914
3,no escopo,Atendimento do vendedor,0.938,0.928,0.913
4,no escopo,Produto diferente do anunciado,0.963,0.936,0.909
5,fora do escopo,Qual é a capital da Austrália?,0.794,0.787,0.780
6,fora do escopo,Receita de bolo de chocolate,0.914,0.887,0.881
7,fora do escopo,Cotação do bitcoin,0.897,0.879,0.874
8,fora do escopo,Escreva uma função em Python,0.890,0.864,0.857
9,fora do escopo,Quem ganhou a Copa de 2002?,0.825,0.805,0.799


**Leitura:** o E5 produz similaridades cosseno altas para qualquer texto (anisotropia): as
distribuições de `top1` dos dois grupos se sobrepõem, e nenhum limiar absoluto separa
"no escopo" de "fora do escopo". Por isso o guardrail usa o classificador de escopo por
exemplos (próxima célula) mais sinais lexicais (ver docs/decisoes_tecnicas.md, seção 10).

## Classificador de escopo: margem entre exemplos do domínio e de fora dele

In [5]:
from voc_rag.generation.scope import ScopeClassifier
scope = ScopeClassifier(embedder)
sim_df["margem_escopo"] = [round(scope.margin(q), 3) for q in sim_df["pergunta"]]
sim_df[["grupo", "pergunta", "top1", "margem_escopo"]]

,grupo,pergunta,top1,margem_escopo
0,no escopo,Problemas com a entrega,0.947,0.040
1,no escopo,Produto veio com defeito,0.978,0.056
2,no escopo,Elogios à qualidade,0.930,0.061
3,no escopo,Atendimento do vendedor,0.938,0.033
4,no escopo,Produto diferente do anunciado,0.963,0.035
5,fora do escopo,Qual é a capital da Austrália?,0.794,-0.050
6,fora do escopo,Receita de bolo de chocolate,0.914,-0.000
7,fora do escopo,Cotação do bitcoin,0.897,0.017
8,fora do escopo,Escreva uma função em Python,0.890,-0.028
9,fora do escopo,Quem ganhou a Copa de 2002?,0.825,-0.026
